In [67]:
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader, Subset, random_split
from torchvision.models import resnet18
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.stats import norm
import requests
import warnings
warnings.filterwarnings('ignore')

#  CONFIGURATION 
N_SHADOW     = 64       # number of shadow models 
EPOCHS       = 50       # epochs per shadow model
BATCH_SIZE   = 256
LR           = 0.1
SHADOW_FRAC  = 0.5      # fraction of pub_ds used per shadow model
N_AUG        = 5        # TTA augmentations for score computation

CHECKPOINT_DIR = Path("./shadow_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BASE       = Path(".")
PUB_PATH   = BASE / "pub.pt"
PRIV_PATH  = BASE / "priv.pt"
MODEL_PATH = BASE / "model.pt"
OUTPUT_CSV = BASE / "submission.csv"


MEAN = [0.7406, 0.5331, 0.7059]
STD  = [0.1491, 0.1864, 0.1301]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
torch.backends.cudnn.benchmark = True

Using device: cuda


In [68]:
from torch.utils.data import Dataset

class TaskDataset(Dataset):
    def __init__(self, transform=None):
        self.ids, self.imgs, self.labels = [], [], []
        self.transform = transform
    def __getitem__(self, index):
        id_ = self.ids[index]
        img = self.imgs[index]
        if self.transform is not None:
            img = self.transform(img)
        return id_, img, self.labels[index]
    def __len__(self):
        return len(self.ids)

class MembershipDataset(TaskDataset):
    def __init__(self, transform=None):
        super().__init__(transform)
        self.membership = []
    def __getitem__(self, index):
        id_, img, label = super().__getitem__(index)
        return id_, img, label, self.membership[index]

transform = T.Compose([
    T.Resize(32),
    T.Normalize(mean=MEAN, std=STD),
])

print("Loading datasets...")
pub_ds  = torch.load(PUB_PATH,  weights_only=False)
priv_ds = torch.load(PRIV_PATH, weights_only=False)
pub_ds.transform  = transform
priv_ds.transform = transform
priv_ds.membership = [-1] * len(priv_ds.ids)

print("Loading target model...")
model = resnet18(weights=None)
model.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
model.maxpool = nn.Identity()
model.fc      = nn.Linear(512, 9)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval().to(device)

pub_loader  = DataLoader(pub_ds,  batch_size=512, shuffle=False, num_workers=0, pin_memory=True)
priv_loader = DataLoader(priv_ds, batch_size=512, shuffle=False, num_workers=0, pin_memory=True)

N_PUB = len(pub_ds)
print(f"pub_ds: {N_PUB} | priv_ds: {len(priv_ds)}")

def make_model():
    m = resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(512, 9)
    return m.to(device)

Loading datasets...
Loading target model...
pub_ds: 14000 | priv_ds: 14000


In [69]:
def compute_logit_scores(net, loader, device, n_aug=N_AUG):
    net.eval()
    all_ids, all_scores, all_labels, all_members = [], [], [], []

    aug_transforms = [
        lambda x: x,                                         
        lambda x: torch.flip(x, dims=[3]),                   
        lambda x: torch.flip(x, dims=[2]),                   
        lambda x: torch.roll(x, shifts=2, dims=3),           
        lambda x: torch.roll(x, shifts=-2, dims=3),          
    ][:n_aug]

    with torch.no_grad():
        for batch in loader:
            if len(batch) == 4:
                ids, imgs, labels, membership = batch
                all_members.extend(membership.tolist())
            else:
                ids, imgs, labels = batch
            ids_list = ids.tolist() if isinstance(ids, torch.Tensor) else ids
            all_ids.extend(ids_list)
            all_labels.extend(labels.tolist())

            imgs, labels = imgs.to(device), labels.to(device)
            aug_scores = []
            for aug_fn in aug_transforms:
                aug_imgs = aug_fn(imgs)
                logits = net(aug_imgs)
                probs  = torch.softmax(logits, dim=1)
                p_y    = probs[torch.arange(len(labels)), labels].clamp(1e-7, 1 - 1e-7)
                log_odds = torch.log(p_y / (1 - p_y))
                aug_scores.append(log_odds)

            score = torch.stack(aug_scores, dim=1).mean(dim=1)
            all_scores.extend(score.cpu().tolist())

    result = {'ids': all_ids, 'scores': np.array(all_scores), 'labels': np.array(all_labels)}
    if all_members:
        result['members'] = np.array(all_members)
    return result

def compute_mentr(net, loader, device, n_aug=N_AUG):
    net.eval()
    all_ids, all_scores, all_members = [], [], []

    aug_fns = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.roll(x, shifts=2, dims=3),
        lambda x: torch.roll(x, shifts=-2, dims=3),
    ][:n_aug]

    with torch.no_grad():
        for batch in loader:
            if len(batch) == 4:
                ids, imgs, labels, membership = batch
                all_members.extend(membership.tolist())
            else:
                ids, imgs, labels = batch

            ids_list = ids.tolist() if isinstance(ids, torch.Tensor) else ids
            all_ids.extend(ids_list)
            imgs, labels = imgs.to(device), labels.to(device)

            aug_scores = []
            for aug_fn in aug_fns:
                aug_imgs  = aug_fn(imgs)
                logits    = net(aug_imgs)
                probs     = torch.softmax(logits, dim=1)
                log_probs = torch.log_softmax(logits, dim=1)

                one_hot = torch.zeros_like(probs)
                one_hot[torch.arange(len(labels)), labels] = 1.0
                mentr = (-(one_hot - probs) * log_probs).sum(dim=1)
                aug_scores.append(-mentr)   # negate: member = low entropy = high score

            score = torch.stack(aug_scores, dim=1).mean(dim=1)
            all_scores.extend(score.cpu().tolist())

    result = {'ids': all_ids, 'scores': np.array(all_scores)}
    if all_members:
        result['members'] = np.array(all_members)
    return result

def lira_score(in_scores_list, out_scores_list, target_score, fix_variance=False):
    if len(in_scores_list) == 0 or len(out_scores_list) == 0:
        return 0.0
    in_arr  = np.array(in_scores_list)
    out_arr = np.array(out_scores_list)
    mu_in  = in_arr.mean()
    mu_out = out_arr.mean()

    if fix_variance:
        pooled_std = np.sqrt(((in_arr.var() * len(in_arr) + out_arr.var() * len(out_arr))
                              / (len(in_arr) + len(out_arr))) + 1e-8)
        std_in = std_out = pooled_std
    else:
        std_in  = max(in_arr.std(),  1e-4)
        std_out = max(out_arr.std(), 1e-4)

    log_p_in  = norm.logpdf(target_score, loc=mu_in,  scale=std_in)
    log_p_out = norm.logpdf(target_score, loc=mu_out, scale=std_out)
    return float(log_p_in - log_p_out)

In [70]:
def train_shadow_and_score(shadow_id, pub_ds, pub_loader, priv_loader, device):
    ckpt_path = CHECKPOINT_DIR / f"shadow_{shadow_id:03d}.pt"
    scores_path = CHECKPOINT_DIR / f"scores_{shadow_id:03d}.npz"

    if scores_path.exists():
        print(f"  [Shadow {shadow_id}] Loading cached scores...")
        data = np.load(scores_path)
        return data['in_mask'], data['pub_scores'], data['priv_scores']

    rng = np.random.RandomState(seed=shadow_id * 1337 + 42)
    indices = rng.permutation(N_PUB)
    n_in    = int(N_PUB * SHADOW_FRAC)
    in_idx  = indices[:n_in]
    in_mask = np.zeros(N_PUB, dtype=bool)
    in_mask[in_idx] = True

    if ckpt_path.exists():
        print(f"  [Shadow {shadow_id}] Loading cached model...")
        shadow = make_model()
        shadow.load_state_dict(torch.load(ckpt_path, map_location=device))
        shadow.eval()
    else:
        shadow_subset = Subset(pub_ds, in_idx.tolist())
        shadow_train_loader = DataLoader(
            shadow_subset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=0, pin_memory=True
        )

        shadow = make_model()
        aug = T.Compose([
            T.RandomHorizontalFlip(),
            T.RandomCrop(32, padding=4, padding_mode='reflect'),
        ])

        optimizer = optim.SGD(shadow.parameters(), lr=LR, momentum=0.9,
                              weight_decay=5e-4, nesterov=True)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        criterion = nn.CrossEntropyLoss()

        shadow.train()
        for epoch in range(EPOCHS):
            for batch in shadow_train_loader:
                if len(batch) == 4:
                    _, imgs, labels, _ = batch
                else:
                    _, imgs, labels = batch
                imgs, labels = imgs.to(device), labels.to(device)
                imgs = aug(imgs)
                optimizer.zero_grad()
                loss = criterion(shadow(imgs), labels)
                loss.backward()
                optimizer.step()
            scheduler.step()
            if (epoch + 1) % 10 == 0:
                print(f"    [Shadow {shadow_id}] Epoch {epoch+1}/{EPOCHS}")

        shadow.eval()
        torch.save(shadow.state_dict(), ckpt_path)

    pub_result  = compute_logit_scores(shadow, pub_loader,  device)
    priv_result = compute_logit_scores(shadow, priv_loader, device)

    pub_scores  = pub_result['scores']
    priv_scores = priv_result['scores']

    np.savez(scores_path, in_mask=in_mask, pub_scores=pub_scores, priv_scores=priv_scores)
    del shadow
    torch.cuda.empty_cache()

    return in_mask, pub_scores, priv_scores

In [71]:
t0 = time.time()

print("\n[Step 1] Computing target model scores...")
target_pub  = compute_logit_scores(model, pub_loader,  device)
target_priv = compute_logit_scores(model, priv_loader, device)

pub_target_scores  = target_pub['scores']   
priv_target_scores = target_priv['scores']  
pub_members        = np.array(target_pub['members'])
pub_ids            = target_pub['ids']
priv_ids           = target_priv['ids']
N_PRIV = len(priv_ids)

print(f"\n[Step 2] Training {N_SHADOW} shadow models...")
pub_in_scores  = np.full((N_PUB,  N_SHADOW), np.nan)
pub_out_scores = np.full((N_PUB,  N_SHADOW), np.nan)
priv_all_scores = np.zeros((N_PRIV, N_SHADOW))

for i in range(N_SHADOW):
    print(f"\n  Shadow model {i+1}/{N_SHADOW}...")
    t1 = time.time()
    in_mask, shadow_pub_scores, shadow_priv_scores = train_shadow_and_score(
        i, pub_ds, pub_loader, priv_loader, device
    )
    pub_in_scores[in_mask, i]   = shadow_pub_scores[in_mask]
    pub_out_scores[~in_mask, i] = shadow_pub_scores[~in_mask]
    priv_all_scores[:, i] = shadow_priv_scores
    elapsed = time.time() - t1
    print(f"  Done in {elapsed:.1f}s | Total elapsed: {(time.time()-t0)/60:.1f} min")

print("\n[Step 3] Computing LiRA scores on pub_ds (local eval)...")
pub_lira_scores = np.zeros(N_PUB)
for j in range(N_PUB):
    in_s  = pub_in_scores[j,  ~np.isnan(pub_in_scores[j])]
    out_s = pub_out_scores[j, ~np.isnan(pub_out_scores[j])]
    pub_lira_scores[j] = lira_score(in_s, out_s, pub_target_scores[j],
                                    fix_variance=(N_SHADOW < 32))

fpr, tpr, _ = roc_curve(pub_members, pub_lira_scores)
auc = roc_auc_score(pub_members, pub_lira_scores)
idx = np.where(fpr <= 0.05)[0][-1]
print(f"  [LOCAL EVAL] LiRA      AUC={auc:.4f}  TPR@5%FPR={tpr[idx]:.4f}")

print("\n[Step 4] Computing modified entropy (secondary signal)...")
pub_mentr  = compute_mentr(model, pub_loader,  device)
priv_mentr = compute_mentr(model, priv_loader, device)
pub_mentr_scores = pub_mentr['scores']
priv_mentr_scores = priv_mentr['scores']
def norm01(x):
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn + 1e-12)

pub_lira_n  = norm01(pub_lira_scores)
pub_mentr_n = norm01(pub_mentr_scores)

best_tpr, best_alpha = 0.0, 0.7
for alpha in np.linspace(0.5, 1.0, 11):
    combined = alpha * pub_lira_n + (1 - alpha) * pub_mentr_n
    fpr_c, tpr_c, _ = roc_curve(pub_members, combined)
    idx_c = np.where(fpr_c <= 0.05)[0][-1]
    if tpr_c[idx_c] > best_tpr:
        best_tpr, best_alpha = tpr_c[idx_c], alpha

print(f"  Best LiRA blend weight: {best_alpha:.2f}  ->  TPR@5%FPR={best_tpr:.4f}")


print("\n[Step 5 - RMIA] Computing RMIA scores...")

def compute_rmia(target_scores, shadow_scores):
    # Calculate the expected score from the population (reference models)
    # Using nanmean to safely ignore NaNs in the pub_out_scores mask
    expected_shadow_scores = np.nanmean(shadow_scores, axis=1)
    
    # RMIA is the ratio of P(target) / P(reference)
    # Since the scores are in log-space, ratio = subtraction
    rmia_scores = target_scores - expected_shadow_scores
    return rmia_scores

# 1. Local Eval on Public Data 
pub_rmia = compute_rmia(pub_target_scores, pub_out_scores)

fpr, tpr, _ = roc_curve(pub_members, pub_rmia)
auc = roc_auc_score(pub_members, pub_rmia)
idx = np.where(fpr <= 0.05)[0][-1]
print(f"  [LOCAL EVAL] pure RMIA   AUC={auc:.4f}  TPR@5%FPR={tpr[idx]:.4f}")

# 2. Final Calculation on Private Data
priv_rmia = compute_rmia(priv_target_scores, priv_all_scores)

# Normalize to 0-1 for submission formatting
def norm01(x):
    mn, mx = x.min(), x.max()
    return (x - mn) / (mx - mn + 1e-12)

priv_rmia_n = norm01(priv_rmia)

print("\n[Step 6] Saving RMIA submission...")
submission_df = pd.DataFrame({
    'id':    [str(i) for i in priv_ids],
    'score': priv_rmia_n
})
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"  Saved {len(submission_df)} rows to {OUTPUT_CSV}")


[Step 1] Computing target model scores...

[Step 2] Training 64 shadow models...

  Shadow model 1/64...
  [Shadow 0] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 2/64...
  [Shadow 1] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 3/64...
  [Shadow 2] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 4/64...
  [Shadow 3] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 5/64...
  [Shadow 4] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 6/64...
  [Shadow 5] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 7/64...
  [Shadow 6] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 8/64...
  [Shadow 7] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min

  Shadow model 9/64...
  [Shadow 8] Loading cached scores...
  Done in 0.0s | Total elapsed: 0.4 min


In [72]:
# Key insights from all our analysis:
# 1. Best blend is equal weights: basic=0.3, full=0.3, perclass=0.3
# 2. Strong classes [1,2,3,7,8] have more signal
# 3. Per-class optimized score adds a tiny bit on top

# Signal 1: RMIA basic
priv_mean_out    = np.nanmean(priv_all_scores, axis=1)
priv_rmia_basic  = priv_target_scores - priv_mean_out

# Signal 2: Full LiRA for priv
all_in_flat   = pub_in_scores[~np.isnan(pub_in_scores)]
all_out_flat  = pub_out_scores[~np.isnan(pub_out_scores)]
global_std    = max(np.concatenate([all_in_flat, all_out_flat]).std(), 1e-4)
mu_in_global  = all_in_flat.mean()
mu_out_global = all_out_flat.mean()

priv_rmia_full = np.zeros(N_PRIV)
for j in range(N_PRIV):
    target_s  = priv_target_scores[j]
    out_s     = priv_all_scores[j]
    mu_out_j  = out_s.mean()
    offset    = mu_out_j - mu_out_global
    log_p_in  = scipy_norm.logpdf(target_s, loc=mu_in_global,           scale=global_std)
    log_p_out = scipy_norm.logpdf(target_s, loc=mu_out_global + offset,  scale=global_std)
    priv_rmia_full[j] = log_p_in - log_p_out

# Signal 3: per-class optimized for priv
priv_labels_arr = np.array([priv_ds[i][2] for i in range(len(priv_ds))])

priv_rmia_perclass = np.zeros(N_PRIV)
for j in range(N_PRIV):
    c              = priv_labels_arr[j]
    class_strength = class_tprs.get(c, 0.05)
    alpha          = np.clip((class_strength - 0.05) / 0.05, 0, 1)
    priv_rmia_perclass[j] = (alpha * priv_rmia_basic[j] + 
                             (1 - alpha) * priv_rmia_full[j])

# blend equally
priv_final = (norm01(priv_rmia_basic) + 
              norm01(priv_rmia_full)  + 
              norm01(priv_rmia_perclass))
priv_final = norm01(priv_final)

# verify on pub first
pub_final = (norm01(rmia_basic)    + 
             norm01(rmia_full)     + 
             norm01(rmia_perclass))
pub_final = norm01(pub_final)

fpr_v, tpr_v, _ = roc_curve(pub_members, pub_final)
a_v   = roc_auc_score(pub_members, pub_final)
idx_v = np.where(fpr_v <= 0.05)[0][-1]
print(f"Verified on pub: AUC={a_v:.4f}  TPR@5%={tpr_v[idx_v]:.4f}")

# save submission
submission_df = pd.DataFrame({
    'id':    [str(i) for i in priv_ids],
    'score': priv_final.astype(float)
})
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(submission_df)} rows to submission.csv")
print(f"Score range: {priv_final.min():.4f} to {priv_final.max():.4f}")

Verified on pub: AUC=0.5069  TPR@5%=0.0653
Saved 14000 rows to submission.csv
Score range: 0.0000 to 1.0000


In [ ]:
import requests
from pathlib import Path

# LEADERBOARD SUBMISSION 
BASE_URL = "YOUR_SERVER_IP_HERE"   
API_KEY  = "YOUR_API_KEY_HERE" 
TASK_ID  = "01-mia"            
OUTPUT_CSV = Path("submission.csv")

def submit_to_leaderboard():
    if not OUTPUT_CSV.exists():
        print(f"Error: Could not find {OUTPUT_CSV.name}. Did the previous cell finish?")
        return

    print(f"Submitting {OUTPUT_CSV.name} to the grading server...")
    try:
        with open(OUTPUT_CSV, "rb") as f:
            resp = requests.post(
                f"{BASE_URL}/submit/{TASK_ID}",
                headers={"X-API-Key": API_KEY},
                files={"file": (OUTPUT_CSV.name, f, "application/csv")},
                timeout=(10, 600), 
            )
        
       
        try:
            body = resp.json()
            print("\nSUCCESS! Server response:")
            print(body)
        except Exception:
            print("\nServer returned a non-JSON response:")
            print(resp.text)
            
        resp.raise_for_status() # Throws an error if we got a 400/500 code
        
    except requests.exceptions.RequestException as e:
        print(f"\nSubmission failed! Error: {e}")

# Fire away
submit_to_leaderboard()

Submitting submission.csv to the grading server...

SUCCESS! Server response:
{'submission_id': 519, 'status': 'success', 'message': 'Submission evaluated successfully! Check the leaderboard to see your score.'}
